# שלב 05 — ניתוח שיבושים ועמידות (Robustness)

זה לב הפרויקט. Centrality אומר מי 'חשוב', אבל האם הסרת תחנה חשובה באמת פוגעת ברשת? מסירים תחנות בזו אחר זו לפי כל אסטרטגיה (Degree / Betweenness / PageRank / Articulation Points) ומול הסרה **אקראית** כקו בסיס, ומודדים איך מתכווץ הרכיב הקשור הגדול (LCC).

אם הרשת קורסת מהר תחת הסרה ממוקדת אבל לאט תחת אקראית — היא פגיעה להתקפה ממוקדת.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx
import random

def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
METRIC_CSV = BASE / "outputs" / "04_centrality_analysis" / "stop_metrics.csv"
AP_CSV = BASE / "outputs" / "03_network_descriptive_analysis" / "articulation_points.csv"
OUT_DIR = BASE / "outputs" / "05_robustness_and_disruption_analysis"
FIG_DIR = BASE / "figures" / "05_robustness_and_disruption_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MAX_REMOVALS = 3000
STEPS = 40
RANDOM_TRIALS = 5
SEED = 42
print("GRAPH_DIR:", GRAPH_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## טעינה

הגרף משלב 02, מדדי המרכזיות משלב 04, ונקודות התורפה משלב 03.

In [ ]:
def load():
    with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
        G = pickle.load(f)
    metrics = pd.read_csv(METRIC_CSV, dtype=str)
    for col in ["degree", "betweenness", "pagerank", "harmonic"]:
        metrics[col] = pd.to_numeric(metrics[col])
    ap_df = pd.read_csv(AP_CSV, dtype=str)
    return G, metrics, ap_df


G, metrics, ap_df = load()
print(f"גרף: {G.number_of_nodes():,} צמתים | AP: {len(ap_df):,}")

## סימולציית ההסרה

`lcc_size` מסיר קבוצת תחנות ומחזיר את גודל הרכיב הגדול שנותר. `simulate` עושה זאת בשלבים לפי סדר נתון. `simulate_random` עושה את אותו דבר עם בחירה אקראית (ממוצע על 5 ניסויים) כקו בסיס.

In [ ]:
def lcc_size(G, removed):
    Gr = G.copy()
    Gr.remove_nodes_from(removed)
    if Gr.number_of_nodes() == 0:
        return 0
    return max(len(c) for c in nx.connected_components(Gr))


def simulate(G, ordered_nodes, max_removals, steps, baseline_n):
    ks = sorted(set(int(round(x)) for x in np.linspace(0, max_removals, steps)))
    rows = []
    for k in ks:
        size = lcc_size(G, set(ordered_nodes[:k]))
        rows.append({"removed": k, "lcc_size": size, "lcc_share": round(size / baseline_n, 4)})
    return rows


def simulate_random(G, max_removals, steps, baseline_n, trials, seed):
    ks = sorted(set(int(round(x)) for x in np.linspace(0, max_removals, steps)))
    rng = random.Random(seed)
    nodes = list(G.nodes())
    rows = []
    for k in ks:
        sizes = [lcc_size(G, set(rng.sample(nodes, min(k, len(nodes))))) for _ in range(trials)]
        rows.append({"removed": k, "lcc_size": np.mean(sizes), "lcc_share": round(np.mean(sizes) / baseline_n, 4)})
    return rows

## הרצת כל האסטרטגיות

**התא הזה עשוי לקחת כמה דקות** (מחשב מחדש את הרכיב הגדול אחרי כל קבוצת הסרות, לכל חמש האסטרטגיות).

In [ ]:
def run_all(G, metrics, ap_df):
    baseline_n = G.number_of_nodes()
    strategies = {
        "degree (גבוה→נמוך)": metrics.sort_values("degree", ascending=False)["stop_id"].tolist(),
        "betweenness (גבוה→נמוך)": metrics.sort_values("betweenness", ascending=False)["stop_id"].tolist(),
        "pagerank (גבוה→נמוך)": metrics.sort_values("pagerank", ascending=False)["stop_id"].tolist(),
        "articulation points": ap_df["stop_id"].tolist() + metrics.sort_values("degree", ascending=False)["stop_id"].tolist(),
    }
    all_results = {}
    for name, ordered in strategies.items():
        print(f"  הסרה לפי: {name}")
        d = pd.DataFrame(simulate(G, ordered, MAX_REMOVALS, STEPS, baseline_n))
        d["strategy"] = name
        all_results[name] = d
    print(f"  הסרה אקראית ({RANDOM_TRIALS} ניסויים)")
    dr = pd.DataFrame(simulate_random(G, MAX_REMOVALS, STEPS, baseline_n, RANDOM_TRIALS, SEED))
    dr["strategy"] = "אקראי (baseline)"
    all_results["אקראי (baseline)"] = dr
    return pd.concat(all_results.values(), ignore_index=True)


df = run_all(G, metrics, ap_df)
df.to_csv(OUT_DIR / "disruption_results.csv", index=False, encoding="utf-8-sig")
print(f"disruption_results.csv: {len(df)} שורות")

## גרף: עקומות העמידות

הגרף המרכזי של הפרויקט. עקומה שיורדת מהר יותר = אסטרטגיה שפוגעת יותר.

In [ ]:
def plot_resilience_curves(df):
    fig, ax = plt.subplots(figsize=(11, 6))
    colors = ["#dc2626", "#2563eb", "#16a34a", "#d97706", "#6b7280"]
    for (strategy, grp), color in zip(df.groupby("strategy"), colors):
        grp = grp.sort_values("removed")
        ax.plot(grp["removed"], grp["lcc_share"], marker="o", markersize=4, linewidth=2, label=strategy, color=color)
    ax.set_xlabel("מספר תחנות שהוסרו")
    ax.set_ylabel("Largest Component Share")
    ax.set_title("עקומות עמידות הרשת לפי אסטרטגיית הסרה")
    ax.set_ylim(max(0.0, df["lcc_share"].min() - 0.03), 1.01)
    ax.legend(loc="lower left", fontsize=9)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "resilience_curves_comparison.png", dpi=150)
    plt.show()


def plot_auc(df):
    auc = df.groupby("strategy").apply(
        lambda g: np.trapz(g.sort_values("removed")["lcc_share"], g.sort_values("removed")["removed"])
    ).reset_index()
    auc.columns = ["strategy", "auc"]
    auc = auc.sort_values("auc")
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(auc["strategy"], auc["auc"], color=["#dc2626", "#2563eb", "#16a34a", "#d97706", "#6b7280"])
    ax.set_xlabel("Area Under Resilience Curve (גבוה = עמיד יותר)")
    ax.set_title("עמידות כוללת לפי אסטרטגיה (AUC)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "auc_comparison.png", dpi=150)
    plt.show()


plot_resilience_curves(df)
plot_auc(df)